In [1]:
import rosbag
import numpy as np
from tqdm import tqdm
from collections import defaultdict
import os

In [2]:
bag_path = "/home/tony/mine/Projects/ArmHandVis/HandVersion/HandArmFiles/ARM_HAND_URDF/banana/banana.bag"
outputpath = "/home/tony/mine/Projects/ArmHandVis/HandVersion/HandArmFiles/ARM_HAND_URDF/banana/TF/"
bag_data = rosbag.Bag(bag_path, "r")
#info = bag_data.get_type_and_topic_info()
bag_data = bag_data.read_messages(topics=['/tf', '/tf_static'])


In [3]:
data_write = defaultdict(list)

link_names = set()
#base on the topic cam0/rgb_image raw
for topic, msg, t in tqdm(bag_data):
    t = np.int64(str(t))
    for transform in msg.transforms:

        frame_id = transform.header.frame_id.replace('/','')
        chind_frame_id = transform.child_frame_id.replace('/','')
        link_names.add(frame_id)
        link_names.add(chind_frame_id)
        dict_key = frame_id + ' -> ' + chind_frame_id
        data_write[dict_key].append([            
            t,
            transform.transform.translation.x,
            transform.transform.translation.y,
            transform.transform.translation.z,
            transform.transform.rotation.x,
            transform.transform.rotation.y,
            transform.transform.rotation.z,
            transform.transform.rotation.w,])


21863it [00:14, 1529.16it/s]


In [4]:
def convert_dict_values_to_float(dictionary):
    for key in dictionary.keys():
        dictionary[key] = [[np.float128(value) for value in list_item] for list_item in dictionary[key]]
convert_dict_values_to_float(data_write)

In [ ]:
def write_dict_to_files(dictionary,write_path):
    path = write_path
    if not os.path.exists(path) :
        os.mkdir(path)  # 创建目录
    for key, value in dictionary.items():
        filename = path + str(key) + ".txt"  # 以键名作为文件名，添加.txt扩展名
        with open(filename, 'w+') as file:
            for sublist in value:
                line = ' '.join(map(str, sublist))  # 将第二层列表中的元素转换为字符串，并以空格分隔
                file.write(line + '\n')  # 写入文件，每行末尾添加换行符
write_dict_to_files(data_write,outputpath)

In [6]:
max_key = ""
max_len = 0
for key,value in data_write.items():
    if len(value) > max_len:
        max_len = len(value)
        max_key = key
print(max_key," ",max_len)

ra_base -> ra_tool0_controller   2535


In [7]:
import rosbag
import pydot

def visualize_tf_tree_from_bag(bag_path, output_path):
    bag = rosbag.Bag(bag_path)
    edges = set()  # 使用集合来存储边，避免重复

    # 读取rosbag中的tf数据
    num = 0
    for _, msg, _ in bag.read_messages(topics=['/tf', '/tf_static']):
        for transform in msg.transforms:
            edges.add((transform.header.frame_id, transform.child_frame_id))
        num += 1
        if num >= 500:
            break
    bag.close()

    # 使用pydot可视化tf tree
    graph = pydot.Dot(graph_type='digraph')

    # 添加所有的边到图中
    for parent, child in edges:
        graph.add_edge(pydot.Edge(parent, child))

    graph.write_pdf(output_path)

output_pdf_path = outputpath + 'tf_tree_visualization.pdf'
visualize_tf_tree_from_bag(bag_path, output_pdf_path)


In [8]:
def find_time_closet(slot,time_stamps):
    diff = np.abs(time_stamps - slot)
    index = np.argmin(diff)
    return index

slot = np.int64(1595)

time_stamps = np.random.randint(0,9999,size = (9,8),dtype=np.int64)
print(time_stamps)
print(find_time_closet(slot,time_stamps[:,0]))

[[4542 1175 8331 8064 9488 1689 6823 7002]
 [1003 4020 7830 5764 3703 9894 7155 1173]
 [9728 4176 3380 3503 4702 5883 6192 9618]
 [1375 5798 7974 5223  797 3104 1263 8339]
 [ 499 3019 9026 9900  669 5596 9901 8356]
 [5921 6023 8750 5819 9145 4721 3341 6270]
 [5348 6093 4039 5543 2405 8419 9995 7325]
 [8945  487 2129 9715 3680 3421 7532 3072]
 [1651 6702 1130 3506 7345 4256 5803 6433]]
8
